# LightBanglaHC author replication

This notebook has not been executed in the author’s Kaggle account. It provides a reproducible independent rerun. Use the complete accompanying repository as a private input. New outputs are written to a separate working directory; original recorded outputs remain intact. CPU execution is intended; attaching GPUs does not accelerate these sklearn estimators. Match Python and package versions for strict comparisons.

In [ ]:
import platform, numpy, scipy, pandas, sklearn
print({"python": platform.python_version(), "numpy": numpy.__version__, "scipy": scipy.__version__, "pandas": pandas.__version__, "sklearn": sklearn.__version__})
# Recorded environment: Python 3.12.14, numpy 2.3.5, scipy 1.17.0, pandas 2.2.3, sklearn 1.8.0.


In [ ]:
from pathlib import Path
Path("audit_scripts").mkdir(exist_ok=True)
Path("audit_scripts/lr_preprocessing_comparison.py").write_text('"""Full-data LR_C3 preprocessing comparison; no historical score is overwritten.\n\nGlobal class weights are held fixed in both arms to isolate TF-IDF fit scope.\nThe current sklearn API omits removed multi_class; SAGA uses multinomial loss\nfor three classes. This is a new environment comparison, not historical replay.\n"""\nimport argparse, ast, gc, hashlib, inspect, json, os, platform, re, signal, sys, time, unicodedata, warnings\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport scipy.sparse as sp\nimport sklearn\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import f1_score\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.utils.class_weight import compute_class_weight\n\ndef load_rules(source_path):\n    tree = ast.parse(Path(source_path).read_text())\n    names = {\'BANGLA_RE\',\'EMOJI_RE\',\'BANGLA_TOXIC\',\'TRANSLIT_TOXIC\',\'ENGLISH_TOXIC\',\'ALL_EXPLICIT\',\'SUBTLE_WORDS\',\'SARCASM_RE\',\'THREAT_RE\',\'HATE_RE\'}\n    funcs = {\'clean\',\'signals\',\'process\',\'num_feats_fast\'}\n    selected = []\n    for node in tree.body:\n        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id in names for t in node.targets): selected.append(node)\n        if isinstance(node, ast.FunctionDef) and node.name in funcs: selected.append(node)\n    env = {\'re\':re,\'unicodedata\':unicodedata,\'np\':np}\n    exec(compile(ast.Module(body=selected,type_ignores=[]), str(source_path), \'exec\'),env)\n    assert funcs.issubset(env)\n    return env\n\ndef feature_views(texts, rules):\n    clean = [rules[\'clean\'](s) for s in texts]\n    sigs = [rules[\'signals\'](s) for s in texts]\n    proc = [c+\' SSEP \'+s for c,s in zip(clean,sigs)]\n    return [proc,proc,clean,sigs]\n\ndef vectorizers():\n    return [TfidfVectorizer(max_features=100000,ngram_range=(1,3),sublinear_tf=True,min_df=1,max_df=.98,analyzer=\'word\',token_pattern=r\'\\S+\',strip_accents=None),\n            TfidfVectorizer(max_features=80000,ngram_range=(2,5),sublinear_tf=True,min_df=2,analyzer=\'char_wb\',strip_accents=None),\n            TfidfVectorizer(max_features=50000,ngram_range=(1,4),sublinear_tf=True,min_df=2,analyzer=\'char\',strip_accents=None),\n            TfidfVectorizer(max_features=10000,ngram_range=(1,2),sublinear_tf=True,min_df=1,analyzer=\'word\',token_pattern=r\'\\S+\')]\n\ndef build_features(views, fit_idx, apply_idx):\n    a,b, widths = [],[],[]\n    for j,(v,raw) in enumerate(zip(vectorizers(),views)):\n        print(\'feature branch\',j+1,\'fit rows\',len(fit_idx),flush=True)\n        x=v.fit_transform([raw[i] for i in fit_idx])\n        z=v.transform([raw[i] for i in apply_idx])\n        widths.append(x.shape[1]);a.append(x);b.append(z)\n    return sp.hstack(a,format=\'csr\'),sp.hstack(b,format=\'csr\'),widths\n\ndef make_lr(cw,C=3.,meta=False):\n    kw=dict(C=C,max_iter=500 if meta else 2000,class_weight=cw,random_state=42)\n    if not meta: kw.update(solver=\'saga\',tol=1e-3)\n    if \'multi_class\' in inspect.signature(LogisticRegression).parameters:kw[\'multi_class\']=\'multinomial\'\n    return LogisticRegression(**kw)\n\ndef main():\n    base=Path(__file__).resolve().parents[2]\n    ap=argparse.ArgumentParser();ap.add_argument(\'--train\',default=str(base/\'data/herwill_2026/train.csv\'));ap.add_argument(\'--source\',default=str(base/\'audit/updated_notebook_source.py\'));ap.add_argument(\'--out\',default=str(Path(__file__).resolve().parents[1]/\'audit/lr_comparison\'));ap.add_argument(\'--wall-seconds\',type=int,default=1800);ap.add_argument(\'--jobs\',type=int,default=1)\n    args=ap.parse_args();out=Path(args.out);out.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();state={\'status\':\'running\',\'environment\':{\'python\':platform.python_version(),\'sklearn\':sklearn.__version__,\'numpy\':np.__version__,\'scipy\':scipy.__version__,\'pandas\':pd.__version__,\'cpu_count\':os.cpu_count()},\'scope\':\'Full-data new-run comparison; historical package versions unknown; global class weights held fixed\',\'arms\':{},\'train_sha256\':hashlib.sha256(Path(args.train).read_bytes()).hexdigest()}\n    def save():\n        state[\'elapsed_seconds\']=time.monotonic()-started\n        (out/\'results.json\').write_text(json.dumps(state,indent=2))\n    def stop(signum,frame):\n        state[\'status\']=\'incomplete_wall_budget\';save();sys.stdout.flush();sys.stderr.flush();os._exit(124)\n    signal.signal(signal.SIGALRM,stop);signal.alarm(args.wall_seconds)\n    save()\n    try:\n        df=pd.read_csv(args.train);texts=df.text.fillna(\'\').astype(str).tolist();y=df.y.to_numpy();rules=load_rules(args.source);views=feature_views(texts,rules)\n        folds=list(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(len(y)),y));cw={int(k):float(v) for k,v in zip([0,1,2],compute_class_weight(\'balanced\',classes=np.array([0,1,2]),y=y))}\n        fold_ids=np.empty(len(y),int)\n        for k,(_,val) in enumerate(folds):fold_ids[val]=k\n        pd.DataFrame({\'id\':df.id,\'y\':y,\'fold\':fold_ids}).to_csv(out/\'folds.csv\',index=False)\n        for arm in [\'global\',\'within_fold\']:\n            print(\'ARM\',arm,flush=True);oof=np.full((len(y),3),np.nan);rec={\'folds\':[],\'completed\':False};state[\'arms\'][arm]=rec;save()\n            if arm==\'global\':\n                mats=[];widths=[]\n                for j,(v,raw) in enumerate(zip(vectorizers(),views)):\n                    print(\'global feature branch\',j+1,flush=True);m=v.fit_transform(raw);widths.append(m.shape[1]);mats.append(m)\n                X=sp.hstack(mats,format=\'csr\');del mats;gc.collect();rec[\'global_widths\']=widths;save()\n            def fit_fold(k,tr,val):\n                if arm==\'global\':a,b=X[tr],X[val];wid=widths\n                else:a,b,wid=build_features(views,tr,val)\n                model=make_lr(cw);t=time.monotonic();print(\'fit\',arm,k+1,a.shape,flush=True)\n                model.fit(a,y[tr])\n                prob=model.predict_proba(b);assert model.classes_.tolist()==[0,1,2]\n                hit_limit=bool(np.any(model.n_iter_>=model.max_iter))\n                fr={\'fold\':k+1,\'macro_f1\':float(f1_score(y[val],prob.argmax(1),average=\'macro\')),\'widths\':wid,\'n_iter\':model.n_iter_.tolist(),\'fit_seconds\':time.monotonic()-t,\'hit_iteration_limit\':hit_limit,\'warnings\':[\'iteration limit reached; inspect convergence\'] if hit_limit else []}\n                print(json.dumps(fr),flush=True);return val,prob,fr\n            with ThreadPoolExecutor(max_workers=args.jobs) as pool:\n                futures=[pool.submit(fit_fold,k,tr,val) for k,(tr,val) in enumerate(folds)]\n                for future in as_completed(futures):\n                    val,prob,fr=future.result();oof[val]=prob;rec[\'folds\'].append(fr);rec[\'folds\'].sort(key=lambda v:v[\'fold\']);save()\n            assert np.isfinite(oof).all();rec[\'pooled_macro_f1\']=float(f1_score(y,oof.argmax(1),average=\'macro\'));rec[\'completed\']=True;np.savez_compressed(out/(arm+\'_oof.npz\'),id=df.id.to_numpy(),y=y,fold=fold_ids,prob=oof);save()\n            if arm==\'global\':del X;gc.collect()\n        state[\'status\']=\'completed\';state[\'global_minus_within_fold\']=state[\'arms\'][\'global\'][\'pooled_macro_f1\']-state[\'arms\'][\'within_fold\'][\'pooled_macro_f1\'];signal.alarm(0);save();print(json.dumps(state),flush=True)\n    except SystemExit:raise\n    except Exception as e:\n        state[\'status\']=\'failed\';state[\'error\']=repr(e);save();raise\n\nif __name__==\'__main__\':main()\n', encoding="utf-8")


In [ ]:
from pathlib import Path
Path("audit_scripts").mkdir(exist_ok=True)
Path("audit_scripts/run_reduced_stack.py").write_text('"""Full-data reduced stacking comparison. No historical results are overwritten.\n\nThree bases: LR_C3, isotonic-calibrated SGD and CNB; nine meta inputs.\nBoth outer assessments use identical base predictions fitted on the outer\ntraining subset. Nested meta-training probabilities are regenerated inside\nthat subset; nonnested training uses the shared outer-OOF matrix.\nAll TF-IDF fits/class weights use the corresponding allowed training subset.\n"""\nimport argparse,gc,hashlib,json,os,platform,time\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor,as_completed\nimport numpy as np,pandas as pd,scipy,sklearn\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.linear_model import LogisticRegression,SGDClassifier\nfrom sklearn.naive_bayes import ComplementNB\nfrom sklearn.metrics import f1_score\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.utils.class_weight import compute_class_weight\nimport lr_preprocessing_comparison as lib\n\n\ndef class_weights(y):\n return dict(zip([0,1,2],map(float,compute_class_weight(\'balanced\',classes=np.arange(3),y=y))))\n\n\ndef fit_bases(a,b,y,widths):\n cw=class_weights(y);start=time.monotonic();lr=lib.make_lr(cw);lr.fit(a,y)\n p=[lr.predict_proba(b)];metadata={\'lr_n_iter\':lr.n_iter_.tolist(),\'lr_cap_reached\':bool((lr.n_iter_>=lr.max_iter).any())}\n sgd=CalibratedClassifierCV(SGDClassifier(loss=\'modified_huber\',alpha=1e-4,max_iter=200,class_weight=cw,random_state=42,n_jobs=1),cv=2,method=\'isotonic\',ensemble=True,n_jobs=1)\n sgd.fit(a,y);p.append(sgd.predict_proba(b))\n char=slice(widths[0],widths[0]+widths[1]+widths[2]);cnb=CalibratedClassifierCV(ComplementNB(alpha=.05,norm=True),cv=2,method=\'isotonic\',ensemble=True,n_jobs=1)\n cnb.fit(a[:,char],y);p.append(cnb.predict_proba(b[:,char]))\n for estimator in [lr,sgd,cnb]:assert estimator.classes_.tolist()==[0,1,2]\n metadata[\'sgd_iterations\']=[c.estimator.n_iter_ for c in sgd.calibrated_classifiers_];metadata[\'fit_seconds\']=time.monotonic()-start\n return np.hstack(p),metadata\n\n\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument(\'--repo\',required=True);ap.add_argument(\'--out\',required=True);ap.add_argument(\'--jobs\',type=int,default=3);args=ap.parse_args();base=Path(args.repo);out=Path(args.out);out.mkdir(parents=True,exist_ok=True)\n tr=pd.read_csv(base/\'data/herwill_2026/train.csv\');y=tr.y.to_numpy();n=len(y);rules=lib.load_rules(base/\'audit/updated_notebook_source.py\');views=lib.feature_views(tr.text.fillna(\'\').astype(str).tolist(),rules);folds=list(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(n),y));fid=np.zeros(n,int)\n for k,(_,e) in enumerate(folds):fid[e]=k\n start=time.monotonic();state={\'status\':\'running\',\'evidence\':\'N: assistant-executed new run; not an author-independent rerun\',\'environment\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'scipy\':scipy.__version__,\'sklearn\':sklearn.__version__},\'train_sha256\':hashlib.sha256((base/\'data/herwill_2026/train.csv\').read_bytes()).hexdigest(),\'script_sha256\':hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\'base_order\':[\'LR_C3\',\'SGD\',\'CNB\'],\'outer_inference\':\'one base fit on all outer training rows, identical outer base probabilities in both arms\',\'feature_and_weight_scope\':\'each permitted base fitting subset; meta weights from outer training labels\',\'outer_folds\':3,\'inner_folds\':3,\'seed\':42,\'base_max_iter\':2000,\'meta_max_iter\':500,\'calibration_cv\':2,\'calibration_ensemble\':True,\'base_records\':[],\'nested_records\':[]}\n def save():state[\'elapsed_seconds\']=time.monotonic()-start;(out/\'reduced_stack_results.json\').write_text(json.dumps(state,indent=2)+\'\\n\')\n save();base_oof=np.full((n,9),np.nan)\n def outer_base(k,t,e):\n  print(\'OUTER BASE\',k+1,flush=True);a,b,widths=lib.build_features(views,t,e);p,meta=fit_bases(a,b,y[t],widths);meta.update(fold=k+1,widths=widths);print(\'OUTER COMPLETE\',k+1,json.dumps(meta),flush=True);return e,p,meta\n with ThreadPoolExecutor(max_workers=args.jobs) as pool:\n  for fut in as_completed([pool.submit(outer_base,k,t,e) for k,(t,e) in enumerate(folds)]):\n   e,p,m=fut.result();base_oof[e]=p;state[\'base_records\'].append(m);save()\n assert np.isfinite(base_oof).all();np.savez_compressed(out/\'reduced_base_oof.npz\',id=tr.id.to_numpy(),y=y,fold=fid,prob=base_oof)\n nonnested=np.full((n,3),np.nan)\n for k,(t,e) in enumerate(folds):\n  meta=lib.make_lr(class_weights(y[t]),C=10,meta=True);meta.fit(base_oof[t],y[t]);nonnested[e]=meta.predict_proba(base_oof[e])\n state[\'nonnested_macro_f1\']=float(f1_score(y,nonnested.argmax(1),average=\'macro\'));print(\'NONNESTED F1\',state[\'nonnested_macro_f1\'],flush=True);save();np.savez_compressed(out/\'reduced_nonnested_oof.npz\',id=tr.id.to_numpy(),y=y,fold=fid,prob=nonnested)\n gc.collect();nested=np.full((n,3),np.nan)\n def outer_nested(k,t,e):\n  print(\'NESTED OUTER\',k+1,flush=True);inner_oof=np.full((len(t),9),np.nan);records=[]\n  inner=list(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(len(t)),y[t]))\n  for j,(uf,vf) in enumerate(inner):\n   u,v=t[uf],t[vf];print(\'INNER\',k+1,j+1,flush=True);a,b,widths=lib.build_features(views,u,v);p,m=fit_bases(a,b,y[u],widths);inner_oof[vf]=p;m.update(inner_fold=j+1,widths=widths);records.append(m);del a,b;gc.collect();print(\'INNER COMPLETE\',k+1,j+1,json.dumps(m),flush=True)\n  assert np.isfinite(inner_oof).all();meta=lib.make_lr(class_weights(y[t]),C=10,meta=True);meta.fit(inner_oof,y[t]);p=meta.predict_proba(base_oof[e]);np.savez_compressed(out/f\'inner_meta_training_outer_{k+1}.npz\',id=tr.id.to_numpy()[t],y=y[t],prob=inner_oof)\n  return e,p,{\'outer_fold\':k+1,\'inner_base_records\':records,\'meta_n_iter\':meta.n_iter_.tolist()}\n with ThreadPoolExecutor(max_workers=args.jobs) as pool:\n  for fut in as_completed([pool.submit(outer_nested,k,t,e) for k,(t,e) in enumerate(folds)]):\n   e,p,m=fut.result();nested[e]=p;state[\'nested_records\'].append(m);save()\n assert np.isfinite(nested).all();state[\'nested_macro_f1\']=float(f1_score(y,nested.argmax(1),average=\'macro\'));state[\'nonnested_minus_nested\']=state[\'nonnested_macro_f1\']-state[\'nested_macro_f1\'];state[\'status\']=\'completed\';save();np.savez_compressed(out/\'reduced_nested_oof.npz\',id=tr.id.to_numpy(),y=y,fold=fid,prob=nested);print(json.dumps(state),flush=True)\n\nif __name__==\'__main__\':main()\n', encoding="utf-8")


In [ ]:
from pathlib import Path
Path("audit_scripts").mkdir(exist_ok=True)
Path("audit_scripts/convergence_sensitivity.py").write_text('"""LBFGS sensitivity for the same C=3 multinomial L2 objective.\n\nDifferent optimizer from historical SAGA; never labelled an exact replay.\nOriginal comparison class weights remain fixed; group/exclusion arms use\nweights from each permitted training fold. Saves fixed OOF predictions.\n"""\nimport argparse,gc,hashlib,json,platform,time\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor,as_completed\nimport numpy as np,pandas as pd,scipy,scipy.sparse as sp,sklearn\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import f1_score\nfrom sklearn.model_selection import StratifiedKFold,StratifiedGroupKFold\nfrom sklearn.utils.class_weight import compute_class_weight\nimport lr_preprocessing_comparison as lib\n\ndef weights(y):return dict(zip([0,1,2],map(float,compute_class_weight(\'balanced\',classes=np.arange(3),y=y))))\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument(\'--repo\',type=Path,required=True);ap.add_argument(\'--out\',type=Path,required=True);ap.add_argument(\'--jobs\',type=int,default=2);ap.add_argument(\'--arms\',nargs=\'+\',default=[\'global\',\'within_fold\',\'cleaner_group\',\'raw_excluded\']);args=ap.parse_args();out=args.out;out.mkdir(parents=True,exist_ok=True)\n df=pd.read_csv(args.repo/\'data/herwill_2026/train.csv\');y=df.y.to_numpy();texts=df.text.fillna(\'\').astype(str).tolist();rules=lib.load_rules(args.repo/\'audit/updated_notebook_source.py\');views=lib.feature_views(texts,rules);cw=weights(y);start=time.monotonic();state={\'status\':\'running\',\'optimizer\':\'LBFGS\',\'C\':3,\'tol\':1e-6,\'max_iter\':2000,\'seed\':42,\'evidence\':\'N: assistant execution, not author-independent replication\',\'environment\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'scipy\':scipy.__version__,\'sklearn\':sklearn.__version__},\'script_sha256\':hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\'arms\':{}}\n def save():state[\'elapsed_seconds\']=time.monotonic()-start;(out/\'convergence_results.json\').write_text(json.dumps(state,indent=2)+\'\\n\')\n for arm in args.arms:\n  keep=np.arange(len(y))\n  if arm==\'raw_excluded\':\n   d=pd.DataFrame({\'text\':texts});fid=np.zeros(len(y),int)\n   for k,(_,v) in enumerate(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(len(y)),y)):fid[v]=k\n   d[\'fold\']=fid;g=d.groupby(\'text\').fold.nunique();keep=np.flatnonzero(~d.text.isin(g[g>1].index).to_numpy())\n  if arm==\'cleaner_group\':splits=list(StratifiedGroupKFold(3,shuffle=True,random_state=42).split(np.zeros(len(keep)),y[keep],groups=pd.factorize(pd.Series(views[2]),sort=False)[0][keep]))\n  else:splits=list(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(len(keep)),y[keep]))\n  splits=[(keep[t],keep[v]) for t,v in splits];rec={\'rows\':len(keep),\'folds\':[],\'class_weight_scope\':\'global fixed\' if arm in [\'global\',\'within_fold\'] else \'training fold\',\'completed\':False};state[\'arms\'][arm]=rec;save();pred=np.full((len(y),3),np.nan);fid=np.full(len(y),-1,int)\n  if arm==\'global\':mats=[v.fit_transform(raw) for v,raw in zip(lib.vectorizers(),views)];X=sp.hstack(mats,format=\'csr\');del mats;gc.collect()\n  def fit(k,t,v):\n   if arm==\'global\':a,b=X[t],X[v];wid=[X.shape[1]]\n   else:a,b,wid=lib.build_features(views,t,v)\n   localcw=cw if arm in [\'global\',\'within_fold\'] else weights(y[t]);model=LogisticRegression(C=3,solver=\'lbfgs\',tol=1e-6,max_iter=2000,class_weight=localcw,random_state=42);now=time.monotonic();print(\'LBFGS\',arm,k+1,a.shape,flush=True);model.fit(a,y[t]);p=model.predict_proba(b)\n   q=model.predict_proba(a);resid=q-np.eye(3)[y[t]];sw=np.array([localcw[int(c)] for c in y[t]]);resid*=sw[:,None]/sw.sum();grad=np.asarray(a.T@resid).T+model.coef_/(3*sw.sum());gradmax=float(max(abs(grad).max(),abs(resid.sum(0)).max()));r={\'fold\':k+1,\'n_iter\':model.n_iter_.tolist(),\'hit_iteration_limit\':bool((model.n_iter_>=model.max_iter).any()),\'penalized_objective_gradient_max_abs\':gradmax,\'macro_f1\':float(f1_score(y[v],p.argmax(1),average=\'macro\')),\'fit_seconds\':time.monotonic()-now,\'widths\':wid};print(json.dumps(r),flush=True);return k,v,p,r\n  with ThreadPoolExecutor(max_workers=args.jobs) as pool:\n   for f in as_completed([pool.submit(fit,k,t,v) for k,(t,v) in enumerate(splits)]):k,v,p,r=f.result();pred[v]=p;fid[v]=k;rec[\'folds\'].append(r);save()\n  assert np.isfinite(pred[keep]).all();rec[\'folds\'].sort(key=lambda x:x[\'fold\']);rec[\'macro_f1\']=float(f1_score(y[keep],pred[keep].argmax(1),average=\'macro\'));rec[\'completed\']=True;np.savez_compressed(out/f\'lbfgs_{arm}_oof.npz\',id=df.id.to_numpy()[keep],y=y[keep],fold=fid[keep],prob=pred[keep]);save()\n  if arm==\'global\':del X;gc.collect()\n state[\'status\']=\'completed\';save();print(json.dumps(state),flush=True)\nif __name__==\'__main__\':main()\n', encoding="utf-8")


In [ ]:
from pathlib import Path
Path("audit_scripts").mkdir(exist_ok=True)
Path("audit_scripts/supporting_checks.py").write_text('"""Deterministic duplicate checks and conditional paired-prediction bootstrap."""\nimport argparse, hashlib, json, sys\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nfrom sklearn.metrics import f1_score\nfrom sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold\nfrom lr_preprocessing_comparison import load_rules\n\ndef f1_from_cm(c):\n    tp=np.diagonal(c,axis1=-2,axis2=-1)\n    den=c.sum(-1)+c.sum(-2)\n    return np.divide(2*tp,den,out=np.zeros_like(tp,dtype=float),where=den!=0).mean(-1)\n\ndef stats(texts,y,fold):\n    df=pd.DataFrame({\'text\':texts,\'y\':y,\'fold\':fold}); groups=df.groupby(\'text\',sort=False).agg(n=(\'y\',\'size\'),labels=(\'y\',\'nunique\'),folds=(\'fold\',\'nunique\'))\n    dup=groups.n>1; cross=dup&(groups.folds>1);conflict=dup&(groups.labels>1)\n    return {\'groups\':int(dup.sum()),\'rows\':int(groups.loc[dup,\'n\'].sum()),\'extra_rows\':int((groups.loc[dup,\'n\']-1).sum()),\'cross_fold_groups\':int(cross.sum()),\'cross_fold_rows\':int(groups.loc[cross,\'n\'].sum()),\'cross_fold_row_percent\':float(100*groups.loc[cross,\'n\'].sum()/len(y)),\'conflicting_groups\':int(conflict.sum()),\'conflicting_rows\':int(groups.loc[conflict,\'n\'].sum()),\'conflicting_row_percent\':float(100*groups.loc[conflict,\'n\'].sum()/len(y)),\'conflicting_cross_fold_groups\':int((conflict&cross).sum())}\n\ndef main():\n    ap=argparse.ArgumentParser();ap.add_argument(\'--repo\',type=Path,required=True);ap.add_argument(\'--out\',type=Path,required=True);ap.add_argument(\'--paired\',type=Path);args=ap.parse_args();args.out.mkdir(parents=True,exist_ok=True)\n    df=pd.read_csv(args.repo/\'data/herwill_2026/train.csv\');y=df.y.to_numpy();raw=df.text.fillna(\'\').astype(str).tolist();clean=[load_rules(args.repo/\'audit/updated_notebook_source.py\')[\'clean\'](t) for t in raw] if False else None\n    rules=load_rules(args.repo/\'audit/updated_notebook_source.py\');clean=[rules[\'clean\'](t) for t in raw];fold=np.empty(len(y),int)\n    for k,(_,v) in enumerate(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(len(y)),y)):fold[v]=k\n    groupfold=np.empty(len(y),int)\n    for k,(_,v) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=42).split(np.zeros(len(y)),y,groups=pd.factorize(pd.Series(clean),sort=False)[0])):groupfold[v]=k\n    pd.DataFrame({\'id\':df.id,\'y\':y,\'stratified_fold\':fold,\'cleaner_group_fold\':groupfold}).to_csv(args.out/\'duplicate_folds.csv\',index=False)\n    raw_frame=pd.DataFrame({\'text\':raw,\'fold\':fold});crossraw=raw_frame.groupby(\'text\').fold.nunique();excluded=raw_frame.text.isin(crossraw[crossraw>1].index).to_numpy()\n    np.savez_compressed(args.out/\'raw_crossfold_exclusion.npz\',id=df.id.to_numpy(),excluded=excluded)\n    result={\'evidence\':\'N: assistant executed; not an author-independent replication\',\'raw\':stats(raw,y,fold),\'notebook_cleaner\':stats(clean,y,fold),\'cleaner_group_split\':stats(clean,y,groupfold),\'excluded_raw_crossfold_rows\':int(excluded.sum()),\'retained_rows\':int((~excluded).sum()),\'group_fold_class_supports\':[[int(np.sum((groupfold==k)&(y==c))) for c in range(3)] for k in range(3)]}\n    old=args.paired or args.repo/\'revision_2026-10-11/audit/lr_comparison\';a=np.load(old/\'global_oof.npz\');b=np.load(old/\'within_fold_oof.npz\');assert np.array_equal(a[\'id\'],df.id) and np.array_equal(a[\'y\'],y) and np.array_equal(a[\'fold\'],fold) and all(np.array_equal(a[k],b[k]) for k in [\'id\',\'y\',\'fold\']);pa=a[\'prob\'].argmax(1);pb=b[\'prob\'].argmax(1);bins=y*9+pa*3+pb;freq=np.bincount(bins,minlength=27).reshape(3,9);rng=np.random.default_rng(42);reps=5000\n    sampled=np.stack([rng.multinomial(int(freq[c].sum()),freq[c]/freq[c].sum(),size=reps) for c in range(3)],axis=1).reshape(reps,3,3,3)\n    delta=f1_from_cm(sampled.sum(3))-f1_from_cm(sampled.sum(2));observed=float(f1_score(y,pa,average=\'macro\')-f1_score(y,pb,average=\'macro\'))\n    result[\'bootstrap\']={\'conditioning\':\'Fixed OOF predictions; training is not resampled. Does not resolve overlapping training folds or user/thread dependence.\',\'global_macro_f1\':float(f1_score(y,pa,average=\'macro\')),\'fold_macro_f1\':float(f1_score(y,pb,average=\'macro\')),\'difference\':observed,\'stratified_row_replicates\':reps,\'seed\':42,\'stratified_row_percentile95\':np.quantile(delta,[.025,.975]).tolist()}\n    codes,_=pd.factorize(pd.Series(raw),sort=False);ng=int(codes.max()+1);mat=sp.csr_matrix((np.ones(len(y),dtype=np.int64),(codes,bins)),shape=(ng,27));cluster_d=[];rng=np.random.default_rng(42)\n    for start in range(0,2000,32):\n        size=min(32,2000-start);idx=rng.integers(0,ng,size=(size,ng));weights=np.stack([np.bincount(row,minlength=ng) for row in idx]);ct=np.asarray(weights@mat).reshape(size,3,3,3);cluster_d.extend((f1_from_cm(ct.sum(3))-f1_from_cm(ct.sum(2))).tolist())\n    result[\'bootstrap\'].update({\'exact_raw_text_cluster_replicates\':2000,\'exact_raw_text_cluster_percentile95\':np.quantile(cluster_d,[.025,.975]).tolist(),\'cluster_unit\':\'Unique exact raw text; conflicting labels remain together; corpus size can change across resamples.\'})\n    newfold=np.empty(int((~excluded).sum()),int);retained=np.flatnonzero(~excluded)\n    for k,(_,v) in enumerate(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(len(retained)),y[retained])):newfold[v]=k\n    result[\'raw_excluded_resplit_duplicates\']=stats([raw[i] for i in retained],y[retained],newfold)\n    result[\'file_hashes\']={str(p.resolve()):hashlib.sha256(p.read_bytes()).hexdigest() for p in [args.repo/\'data/herwill_2026/train.csv\',args.repo/\'audit/updated_notebook_source.py\',old/\'global_oof.npz\',old/\'within_fold_oof.npz\']}\n    (args.out/\'supporting_checks.json\').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))\nif __name__==\'__main__\':main()\n', encoding="utf-8")


In [ ]:
from pathlib import Path
Path("audit_scripts").mkdir(exist_ok=True)
Path("audit_scripts/exclusion_fixed_folds.py").write_text('"""Exclude original raw cross-fold groups while preserving original fold IDs."""\nimport argparse,json,time,hashlib,gc\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor,as_completed\nimport numpy as np,pandas as pd\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.utils.class_weight import compute_class_weight\nfrom sklearn.metrics import f1_score\nimport lr_preprocessing_comparison as lib\np=argparse.ArgumentParser();p.add_argument(\'--repo\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True);p.add_argument(\'--jobs\',type=int,default=2);args=p.parse_args();args.out.mkdir(parents=True,exist_ok=True);df=pd.read_csv(args.repo/\'data/herwill_2026/train.csv\');y=df.y.to_numpy();raw=df.text.fillna(\'\').astype(str).tolist();fold=np.empty(len(y),int)\nfor k,(_,v) in enumerate(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(len(y)),y)):fold[v]=k\ng=pd.DataFrame({\'text\':raw,\'fold\':fold}).groupby(\'text\').fold.nunique();exclude=pd.Series(raw).isin(g[g>1].index).to_numpy();keep=np.flatnonzero(~exclude);assert len(keep)==47046;assert pd.DataFrame({\'text\':np.array(raw,dtype=object)[keep],\'fold\':fold[keep]}).groupby(\'text\').fold.nunique().max()==1\nviews=lib.feature_views(raw,lib.load_rules(args.repo/\'audit/updated_notebook_source.py\'));pred=np.full((len(y),3),np.nan);start=time.monotonic();state={\'status\':\'running\',\'evidence\':\'N: assistant execution; not author-independent replication\',\'policy\':\'Remove all 771 rows in original spanning raw-text groups; retain original seed-42 stratified fold IDs on remaining rows. No resplit.\',\'rows\':len(keep),\'excluded_rows\':int(exclude.sum()),\'raw_cross_fold_groups_after_exclusion\':0,\'C\':3,\'solver\':\'lbfgs\',\'tol\':1e-6,\'max_iter\':2000,\'class_weight_scope\':\'per retained training subset\',\'folds\':[],\'script_sha256\':hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}\ndef save():state[\'elapsed_seconds\']=time.monotonic()-start;(args.out/\'exclusion_fixedfold_results.json\').write_text(json.dumps(state,indent=2)+\'\\n\')\ndef fit(k):\n t=keep[fold[keep]!=k];v=keep[fold[keep]==k];a,b,w=lib.build_features(views,t,v);cw=dict(zip([0,1,2],map(float,compute_class_weight(\'balanced\',classes=np.arange(3),y=y[t]))));m=LogisticRegression(C=3,solver=\'lbfgs\',tol=1e-6,max_iter=2000,class_weight=cw,random_state=42);now=time.monotonic();m.fit(a,y[t]);prob=m.predict_proba(b);res=m.predict_proba(a)-np.eye(3)[y[t]];sw=np.array([cw[int(c)] for c in y[t]]);res*=sw[:,None]/sw.sum();grad=np.asarray(a.T@res).T+m.coef_/(3*sw.sum());record={\'fold\':k+1,\'n_iter\':m.n_iter_.tolist(),\'hit_iteration_limit\':bool((m.n_iter_>=m.max_iter).any()),\'penalized_objective_gradient_max_abs\':float(max(abs(grad).max(),abs(res.sum(0)).max())),\'fit_seconds\':time.monotonic()-now,\'rows_train\':len(t),\'rows_assessment\':len(v),\'widths\':w};print(json.dumps(record),flush=True);return v,prob,record\nsave()\nwith ThreadPoolExecutor(max_workers=args.jobs) as pool:\n for f in as_completed([pool.submit(fit,k) for k in range(3)]):v,prob,record=f.result();pred[v]=prob;state[\'folds\'].append(record);save()\nassert np.isfinite(pred[keep]).all();state[\'folds\'].sort(key=lambda x:x[\'fold\']);state[\'macro_f1\']=float(f1_score(y[keep],pred[keep].argmax(1),average=\'macro\'));state[\'status\']=\'completed\';np.savez_compressed(args.out/\'lbfgs_exclusion_fixedfold_oof.npz\',id=df.id.to_numpy()[keep],y=y[keep],fold=fold[keep],prob=pred[keep]);save();print(json.dumps(state),flush=True)\n', encoding="utf-8")


In [ ]:
from pathlib import Path
Path("audit_scripts").mkdir(exist_ok=True)
Path("audit_scripts/run_all.py").write_text('"""Run auditable comparisons sequentially to limit peak memory."""\nimport argparse, subprocess,sys\nfrom pathlib import Path\np=argparse.ArgumentParser();p.add_argument(\'--repo\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True);p.add_argument(\'--jobs\',type=int,default=1);a=p.parse_args();scripts=Path(__file__).resolve().parent\ncommands=[[\'lr_preprocessing_comparison.py\',\'--train\',str(a.repo/\'data/herwill_2026/train.csv\'),\'--source\',str(a.repo/\'audit/updated_notebook_source.py\'),\'--out\',str(a.out/\'paired_saga\'),\'--wall-seconds\',\'14400\',\'--jobs\',str(a.jobs)],[\'run_reduced_stack.py\',\'--repo\',str(a.repo),\'--out\',str(a.out),\'--jobs\',str(a.jobs)],[\'convergence_sensitivity.py\',\'--repo\',str(a.repo),\'--out\',str(a.out),\'--jobs\',str(a.jobs)],[\'exclusion_fixed_folds.py\',\'--repo\',str(a.repo),\'--out\',str(a.out),\'--jobs\',str(a.jobs)],[\'supporting_checks.py\',\'--repo\',str(a.repo),\'--out\',str(a.out),\'--paired\',str(a.out/\'paired_saga\')]]\nfor cmd in commands:subprocess.run([sys.executable,str(scripts/cmd[0]),*cmd[1:]],check=True)\n', encoding="utf-8")


In [ ]:
from pathlib import Path
import shutil
roots = [Path.cwd(), Path("/kaggle/input")]
candidates = [p.parent.parent for root in roots if root.exists() for p in root.rglob("audit/updated_notebook_source.py") if (p.parent.parent / "data/herwill_2026/train.csv").exists()]
if not candidates:
    raise FileNotFoundError("Attach the full repository input with audit/updated_notebook_source.py and data/herwill_2026/train.csv.")
SOURCE_REPO = candidates[0]
REPO = Path("independent_replication_repo").resolve()
(REPO / "audit").mkdir(parents=True, exist_ok=True)
(REPO / "data/herwill_2026").mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE_REPO / "audit/updated_notebook_source.py", REPO / "audit/updated_notebook_source.py")
shutil.copy2(SOURCE_REPO / "data/herwill_2026/train.csv", REPO / "data/herwill_2026/train.csv")
OUT = Path("independent_replication_results").resolve()
print("Source:", SOURCE_REPO, "Output:", OUT)


In [ ]:
import os, subprocess, sys
env = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1")
subprocess.run([sys.executable, "audit_scripts/run_all.py", "--repo", str(REPO), "--out", str(OUT), "--jobs", "1"], env=env, check=True)


In [ ]:
import json
from pathlib import Path
paired = json.loads((OUT / "paired_saga/results.json").read_text())
print("global", paired["arms"]["global"]["pooled_macro_f1"])
print("within-fold", paired["arms"]["within_fold"]["pooled_macro_f1"])
print("Recorded comparison: 0.5220608956637554 vs 0.5188715615569914")
for name in ["reduced_stack_results.json", "convergence_results.json", "exclusion_fixedfold_results.json", "supporting_checks.json"]:
    print(name, json.loads((OUT / name).read_text()))


Retain the executed notebook, stdout, JSON records and probability files. Save screenshots only from real runs. If scores differ, preserve the discrepancy and environment; do not edit predictions or substitute target values. The fixed-prediction bootstrap is conditional on fitted predictions and does not resample training. A Kaggle run is author-independent only if the authors actually execute and review it.